# SaProt-35M + LoRA DTI Training on Colab T4

**목표:** DAVIS 데이터셋으로 SaProt-35M + LoRA 파인튜닝 → Pearson r ≥ 0.85 달성

## 사전 준비
1. 런타임 → 런타임 유형 변경 → **T4 GPU** 선택
2. 상단 메뉴에서 런타임 연결 확인 (RAM / Disk 표시되면 정상)
3. 아래 셀을 순서대로 실행

## Step 1. GPU 확인

In [1]:
!nvidia-smi
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

Thu Mar 26 05:58:40 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   42C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## Step 2. 패키지 설치

> **주의:** torchvision / torchaudio 설치 금지 (transformers 충돌)

In [2]:
# 1) rdkit 먼저 설치 (DeepPurpose가 구버전 rdkit-pypi 참조 → 충돌 방지)
!pip install -q rdkit

# 2) DeepPurpose: --no-deps로 의존성 무시하고 설치
!pip install -q --no-deps git+https://github.com/kexinhuang12345/DeepPurpose.git

# 3) 나머지 패키지 별도 설치
!pip install -q \
    "transformers>=4.36.0" \
    "peft>=0.9.0" \
    "accelerate>=0.24.0" \
    "bitsandbytes>=0.41.0" \
    "scipy>=1.10.0" \
    "pandas>=2.0.0" \
    "matplotlib>=3.7.0" \
    "scikit-learn" \
    "wget" \
    "subword-nmt" \
    "PyBioMed" \
    "tape_proteins"

# 설치 확인
import rdkit
from DeepPurpose import dataset
print(f"✅ rdkit {rdkit.__version__}")
print("✅ DeepPurpose OK")
print("✅ 설치 완료")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.7/36.7 MB 62.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 19.7 MB/s eta 0:00:00


ModuleNotFoundError: No module named 'wget'

## Step 3. GitHub에서 코드 가져오기

In [ ]:
import os

REPO_URL = "https://github.com/ohsejun97/Capstone-Design.git"  # 본인 레포 URL
REPO_DIR = "/content/Capstone_Design"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull

os.chdir(REPO_DIR)
print(f"Working directory: {os.getcwd()}")
!ls

## Step 4. Google Drive 마운트 (결과 저장용)

> 런타임 종료 후에도 결과를 보존하려면 실행. 건너뛰면 `/content/` 에만 저장됨.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# 결과 저장 디렉토리
SAVE_DIR = "/content/drive/MyDrive/CapstoneResults"
os.makedirs(SAVE_DIR, exist_ok=True)
print(f"결과 저장 위치: {SAVE_DIR}")

## Step 5. 학습 실행 — SaProt-35M + LoRA

### 예상 소요 시간 (T4 기준)
| 항목 | 값 |
|------|----|
| 에폭당 | ~10~15분 |
| 50 에폭 (early stopping 없을 때) | ~8~12시간 |
| Early stopping (patience=10) | 보통 20~30 에폭에서 종료 → ~4~6시간 |
| 학습 파라미터 | 0.55M / 34.3M (1.6%) |

> Colab 무료 버전은 최대 12시간 세션 — Drive 마운트로 중간 체크포인트 저장 권장

In [ ]:
# LoRA 학습 실행
!python train_dti_saprot.py \
    --encoder 35M \
    --lora \
    --epochs 50 \
    --patience 10 \
    --seed 42

## Step 6. 결과 확인

In [ ]:
import json, glob

# 결과 파일 찾기
result_files = sorted(glob.glob("results/*/metrics.json"))
for f in result_files:
    with open(f) as fp:
        m = json.load(fp)
    name = f.split('/')[1]
    print(f"{name:40s} | test_r={m.get('test_pearson_r', '?'):.4f} | val_r={m.get('best_val_r', '?'):.4f}")

In [ ]:
import matplotlib.pyplot as plt
import json, glob

result_files = sorted(glob.glob("results/*/metrics.json"))
if result_files:
    f = result_files[-1]  # 가장 최근 결과
    with open(f) as fp:
        m = json.load(fp)

    # val_history가 있으면 학습 곡선 그리기
    history_file = f.replace("metrics.json", "val_history.json")
    if os.path.exists(history_file):
        with open(history_file) as fp:
            history = json.load(fp)
        plt.figure(figsize=(10, 4))
        plt.subplot(1, 2, 1)
        plt.plot(history['train_loss'], label='Train Loss')
        plt.xlabel('Epoch'); plt.ylabel('HuberLoss'); plt.title('Train Loss')
        plt.legend()
        plt.subplot(1, 2, 2)
        plt.plot(history['val_r'], label='Val Pearson r', color='orange')
        plt.axhline(0.7855, linestyle='--', color='gray', label='650M Frozen (0.7855)')
        plt.xlabel('Epoch'); plt.ylabel('Pearson r'); plt.title('Validation Pearson r')
        plt.legend()
        plt.tight_layout()
        plt.savefig('results/training_curve.png', dpi=150)
        plt.show()
        print("곡선 저장: results/training_curve.png")
    else:
        print(f"최종 결과: {m}")

## Step 7. 결과를 Google Drive에 복사

In [ ]:
import shutil

# results 폴더 전체를 Drive에 복사
src = "/content/Capstone_Design/results"
dst = f"{SAVE_DIR}/results"
if os.path.exists(src):
    shutil.copytree(src, dst, dirs_exist_ok=True)
    print(f"✅ 결과 복사 완료: {dst}")

# lora_adapter.pt 별도 복사 (용량 주의: ~2MB)
adapter_files = glob.glob("/content/Capstone_Design/results/*/lora_adapter.pt")
for af in adapter_files:
    name = af.split('/')[-2]
    dst_af = f"{SAVE_DIR}/{name}_lora_adapter.pt"
    shutil.copy2(af, dst_af)
    print(f"  어댑터 저장: {dst_af}")

## 참고: 경량화 비교 실험 설계

| 실험 | 명령어 | 예상 r | 예상 학습 시간 (T4) |
|------|--------|--------|--------------------|
| SaProt-650M frozen (기준) | `--encoder 650M` | 0.7855 (완료) | ~1분 (로컬) |
| SaProt-35M frozen | `--encoder 35M` | 0.7832 (완료) | ~1분 (로컬) |
| **SaProt-35M + LoRA** ← 현재 | `--encoder 35M --lora` | 0.85~0.92 목표 | ~4~6h |
| SaProt-650M + LoRA | `--encoder 650M --lora` | 0.88~0.93 목표 | ~8~12h |

**핵심 연구 질문:**
- 35M + LoRA가 frozen 650M을 초과하는가? (초과 시 경량화 성공)
- 파라미터 18배 작은 모델이 LoRA만으로 대형 frozen 모델을 따라잡을 수 있는가?